**Disclaimer**: Much of this notebook was generated with the aid of Claude Opus 5.5

# Reidemeister invariance of Bar-Natan multifactorizations

Companion to the invariance proof. One section per move. Each builds the cube of
resolutions, applies the preparatory isomorphisms of Step 1, the special
deformation retracts of Step 2 and the Gaussian eliminations of Step 3, prints
the resulting differentials, and checks that what comes out is a matrix
multifactorization: $d_0^2 = W$ at every vertex, $\mu_1(d_1) = 0$, and
$d_1d_1 + \mu_1(d_2) = 0$.

**Conventions.** Arcs are numbered from $1$, as in the paper: `T_i` and `D_i`
are the Koszul operators $\theta_i$ and $\partial_i$ of arc $i$. A crossing is
`[e0, e1, e2, e3, c, s]` — its four edges, the colour, and the sign; the $i$-th
crossing of the list carries arc $i$. `X_e` is the edge variable of edge `e`
(edges keep their own numbering, so the boundary variables are $x_0,\dots,x_4$),
`G` is the deformation parameter, and the edge passed as `u_edge` plays the role
of $u$. A cube vertex is its tuple of resolution bits; a delooped vertex splits
in two, recorded by an extra bit, so `(1,0,1,0)` and `(1,0,1,1)` are the $u^0$
and $u^1$ summands over `(1,0,1)`. Finally `simplification_type[v] = (a, deg)`
names the arc simplified at `v` and whether by arc removal (`deg=1`) or
delooping (`deg=2`). Which arc is called $\alpha_i$ in the figures is fixed by
the label permutations of Step 1 and is cosmetic.

## Setup

The matrix-factorization engine, the cube construction, the perturbation and
cancellation machinery, and the verification routines.

In [1]:
import functools
import itertools
import re
import copy
import sympy as sp

# ======================================================================
# 1.  Words in the Koszul operators
#
# T_i and D_i are the multiplication and contraction operators of arc i,
# written theta_{i+1} and partial_{i+1} in the paper.  They satisfy
# T_i^2 = D_i^2 = 0, D_i T_i + T_i D_i = 1, and anticommute across
# different indices.  `group_and_factor` puts any expression into the
# normal form "coefficient times increasing word in T_1, D_1, T_2, ...".
# ======================================================================

_OP = re.compile(r'^[DT]_\d+$')

def _is_operator(x):
    return isinstance(x, sp.Symbol) and bool(_OP.match(x.name))

def T(a):
    """Multiplication operator of arc a.  Arcs are numbered from 1, as in the paper."""
    return sp.Symbol(f"T_{a}", commutative=False)

def D(a):
    """Contraction operator of arc a."""
    return sp.Symbol(f"D_{a}", commutative=False)

def parse_nc_term(term):
    """Split a monomial into (commutative coefficient, flat list of operators)."""
    if term == 0:
        return sp.S.Zero, []
    args = term.args if term.func == sp.Mul else [term]
    coeff, ops = [], []
    for a in args:
        if a.func == sp.Pow and _is_operator(a.base):
            ops.extend([a.base] * int(a.exp))
        elif _is_operator(a):
            ops.append(a)
        else:
            coeff.append(a)
    return (sp.Mul(*coeff) if coeff else sp.S.One), ops

def _priority(op):
    """T_i before D_i, and index i before index i+1."""
    return 2 * int(op.name[2:]) + (1 if op.name[0] == 'D' else 0)

def normalize_term(coeff, ops):
    """Rewrite coeff * ops in the normal form, branching at each D_i T_i."""
    if coeff == 0:
        return 0
    stack, out = [(coeff, list(ops))], 0
    while stack:
        c, w = stack.pop()
        if c == 0:
            continue
        for k in range(len(w) - 1):
            A, B = w[k], w[k + 1]
            if A == B:                                  # T_i^2 = D_i^2 = 0
                break
            if _priority(A) <= _priority(B):
                continue
            if A.name[0] == 'D' and B.name[0] == 'T' and A.name[2:] == B.name[2:]:
                stack.append((c, w[:k] + w[k + 2:]))            # D_i T_i = 1 - T_i D_i
                stack.append((-c, w[:k] + [B, A] + w[k + 2:]))
            else:
                stack.append((-c, w[:k] + [B, A] + w[k + 2:]))  # anticommute
            break
        else:
            out += c * sp.Mul(*w) if w else c
    return out

def group_and_factor(expr):
    """Normal form: every monomial reduced, then grouped and the coefficients factored."""
    if expr == 0:
        return sp.S.Zero
    grouped = {}
    for term in sp.Add.make_args(sp.expand(expr)):
        c, ops = parse_nc_term(term)
        for st in sp.Add.make_args(sp.expand(normalize_term(c, ops))):
            sc, sops = parse_nc_term(st)
            key = tuple(sops)
            grouped[key] = grouped.get(key, 0) + sc
    out = 0
    for ops, coeff in grouped.items():
        if coeff != 0:
            out += sp.factor(sp.simplify(coeff)) * (sp.Mul(*ops) if ops else 1)
    return sp.S(out)

def hom_differential(f, d_M, d_N, deg=None):
    """mu_1(f) = d_N f - (-1)^deg f d_M, with deg read off f term by term if omitted."""
    out = 0
    for term in sp.Add.make_args(sp.expand(f)):
        _, ops = parse_nc_term(term)
        e = (len(ops) if deg is None else deg) % 2
        for rt in sp.Add.make_args(sp.expand(d_N * term - (-1) ** e * term * d_M)):
            rc, rops = parse_nc_term(rt)
            out += normalize_term(rc, rops)
    return group_and_factor(out)

# ======================================================================
# 2.  Division of coefficients with respect to the internal variable u
#
# poly_quo is denominator free: it stops as soon as a leading coefficient
# fails to divide over Z, so a quotient it returns is exact in that range
# and the caller sees the remainder through Quot / P.
# ======================================================================

def poly_quo(f, g, x):
    """
    Quotient of f by g with respect to x, over Z.  Every degenerate case raises
    rather than returning a partial answer: a silently truncated quotient is
    exactly how a wrong simplification choice would go unnoticed.
    """
    if f == 0:
        return sp.S.Zero
    if g == 0:
        raise ZeroDivisionError("poly_quo: zero divisor")
    syms = set().union(*(e.free_symbols for e in (f, g, x) if isinstance(e, sp.Basic)))
    to_comm = {s: sp.Symbol(s.name, commutative=True) for s in syms}
    f, g, x = (e.xreplace(to_comm) for e in (f, g, x))
    f, g = sp.expand(f), sp.expand(g)
    deg_g = sp.degree(g, x)
    if deg_g == 0:
        raise ValueError(f"poly_quo: the divisor {g} does not involve {x}; "
                         "the arc chosen for simplification cannot be the right one")
    lc_g, q, rem = sp.LC(g, x), sp.S.Zero, f
    while rem != 0 and sp.degree(rem, x) >= deg_g:
        c = sp.cancel(sp.LC(rem, x) / lc_g)
        if sp.together(c).as_numer_denom()[1] != 1:
            raise ValueError(f"poly_quo: {lc_g} does not divide {sp.LC(rem, x)} over Z, "
                             "so the quotient would leave the boundary ring")
        term = c * x ** (sp.degree(rem, x) - deg_g)
        q, rem = q + term, sp.expand(rem - term * g)
    inv = {v: k for k, v in to_comm.items()}
    return sp.factor(sp.simplify(q)).xreplace(inv)

def _map_coeffs(A, fn):
    """Group A by operator word and apply fn to each coefficient."""
    if A == 0:
        return sp.S.Zero
    grouped = {}
    for term in sp.Add.make_args(sp.expand(A)):
        c, ops = parse_nc_term(term)
        key = tuple(ops)
        grouped[key] = grouped.get(key, 0) + c
    out = 0
    for ops, coeff in grouped.items():
        if coeff == 0:
            continue
        new = fn(sp.expand(coeff))
        if new != 0:
            out += sp.factor(sp.simplify(new)) * (sp.Mul(*ops) if ops else 1)
    return out

def Quot(A, p, u):
    """Quo^u_p applied coefficientwise."""
    return _map_coeffs(A, lambda c: poly_quo(c, p, u))

def P(A, p, u, k):
    """P^{u,k}_p: the coefficient of u^k in the remainder of each coefficient mod p."""
    def take(c):
        rem = sp.expand(c - poly_quo(c, p, u) * p)
        return 0 if rem == 0 else (rem.subs(u, 0) if k == 0 else rem.coeff(u, int(k)))
    return _map_coeffs(A, take)

# ======================================================================
# 3.  Decorated diagrams and their matrix factorizations
#
# arc_data is a list of arcs [e0, e1, e2, e3, c]: the four adjacent edges
# and the colour.  c = 2 marks an arc removed by an arc removal and c = 3
# one removed by a delooping; such an arc contributes nothing to d_0.
# ======================================================================

class SymbolicMatrixFactorization:
    def __init__(self, arc_data):
        self.arc_data = arc_data
        self.G = sp.Symbol('G')
        self.iso = 1            # isomorphism onto this diagram, accumulated
        self.iso_inv = 1
        self.edge_ids = {e for arc in arc_data for e in arc[:4]}
        self.edges = {e: sp.Symbol(f"X_{e}") for e in sorted(self.edge_ids)}

    @property
    def arcs(self):
        """The arc numbers 1, ..., r of this diagram."""
        return range(1, len(self.arc_data) + 1)

    def _arc(self, a):
        return self.arc_data[a - 1]

    def _edges_of(self, a):
        x, y, z, w = (self.edges[e] for e in self._arc(a)[:4])
        return x, y, z, w, self._arc(a)[4]

    def get_linear(self, a):
        """a_a, the linear entry of the Koszul matrix (0 for a removed arc)."""
        x, y, z, w, c = self._edges_of(a)
        return 0 if c > 1 else (-1) ** c * (y - z)

    def get_quadratic(self, a):
        """b_a, the quadratic entry of the Koszul matrix (0 for a removed arc)."""
        x, y, z, w, c = self._edges_of(a)
        return 0 if c > 1 else (y - x) * (x + z + self.G)

    def get_custom_ring_reduction(self, generators):
        """Solve the edge relations for the non-generating edge variables."""
        relations = []
        for arc in self.arc_data:
            x, y, z, w = (self.edges[e] for e in arc[:4])
            relations.append(x - y + z - w)
            if arc[4] == 2:
                relations.append(x - w)
            elif arc[4] == 3:
                relations.append(x - y)
        unknowns = list(set(self.edges.values()) - set(generators))
        res = sp.solve(relations, unknowns, dict=True)
        return res[0] if res else {}

    def get_differential(self, generators):
        """d_0 = sum_i (a_i D_i + b_i T_i), in the given generators."""
        d = sum(self.get_linear(a) * D(a) + self.get_quadratic(a) * T(a)
                for a in self.arcs)
        return group_and_factor(d).subs(self.get_custom_ring_reduction(generators))

    def get_inclusion_operator(self, j, edge_id, param=1, subs_dict=None):
        """
        The inclusion iota of the special deformation retract that simplifies
        arc j: by arc removal (param=1, divisor a_j) or delooping (param=2,
        divisor b_j).  Divisions are with respect to the edge variable
        `edge_id`, in the reduced boundary ring when subs_dict is given.
        """
        if param not in (1, 2):
            raise ValueError("param must be 1 (arc removal) or 2 (delooping)")
        u = self.edges[edge_id]
        divisor = self.get_linear(j) if param == 1 else self.get_quadratic(j)
        if subs_dict:
            divisor = sp.expand(divisor.subs(subs_dict))
            u = u.subs(subs_dict)
            if not isinstance(u, sp.Symbol):
                raise ValueError(f"X_{edge_id} was eliminated in the ring reduction; "
                                 "pick one of the independent generators")
        out = 0
        for i in self.arcs:
            a, b = self.get_linear(i), self.get_quadratic(i)
            if subs_dict:
                a, b = sp.expand(a.subs(subs_dict)), sp.expand(b.subs(subs_dict))
            quo_a, quo_b = poly_quo(a, divisor, u), poly_quo(b, divisor, u)
            if param == 1:
                out += quo_a * D(i) * T(j)
                if i != j:
                    out += quo_b * T(i) * T(j)
            else:
                if i != j and sp.degree(sp.expand(a - divisor * quo_a), u) > 0:
                    raise ValueError(f"a_{i} does not satisfy the residue condition")
                if sp.degree(sp.expand(b - divisor * quo_b), u) > 0:
                    raise ValueError(f"b_{i} does not satisfy the residue condition")
                out += quo_a * D(i) + quo_b * T(i)
        return out

    def get_reduction(self, j, param):
        """
        The diagram with arc j removed: by arc removal (1) or delooping (2).
        A delooping re-bases the remaining arcs; arc j keeps its own edge tuple
        and only its colour changes, which is how it is drawn in the figures.
        """
        arcs = copy.deepcopy(self.arc_data)
        arc_j = arcs[j - 1][:4]
        if param == 2:
            arcs = [[z, w, x, y, c] for x, y, z, w, c in arcs]
        arcs[j - 1] = arc_j + [2 if param == 1 else 3]
        return SymbolicMatrixFactorization(arcs)

    def _transformed(self, arcs):
        out = SymbolicMatrixFactorization(arcs)
        out.iso, out.iso_inv = self.iso, self.iso_inv
        return out

    def flip_arrows(self, arcs_to_flip=None):
        """
        Reverse the given arcs (all of them by default); the arc reversal
        s_a = d_a t_a - t_a d_a of each one.
        """
        arcs = copy.deepcopy(self.arc_data)
        out = self._transformed(arcs)
        for a in (self.arcs if arcs_to_flip is None else arcs_to_flip):
            x, y, z, w, c = arcs[a - 1]
            arcs[a - 1] = [z, w, x, y, c]
            s = D(a) * T(a) - T(a) * D(a)
            out.iso, out.iso_inv = s * out.iso, out.iso_inv * s
        out.arc_data = arcs
        return out

    def arc_relabel(self, relabel_dict):
        """Exchange the labels of arcs i and j; the label permutation tau_{i,j}."""
        arcs = copy.deepcopy(self.arc_data)
        out = self._transformed(arcs)
        for i, j in relabel_dict.items():
            arcs[i - 1], arcs[j - 1] = arcs[j - 1], arcs[i - 1]
            tau = (T(i) * D(j) + T(j) * D(i)
                   + T(j) * T(i) * D(j) * D(i) + D(j) * D(i) * T(i) * T(j))
            out.iso, out.iso_inv = tau * out.iso, out.iso_inv * tau
        out.arc_data = arcs
        return out

    def arc_slide(self, i, j, common_edge):
        """
        Slide arc i along arc j across their shared edge; the arc slide psi_{i,j}.
        Note the order: arc i moves, arc j does not, so the arguments do not
        commute.  Only the branch used below (pos[0] - pos[1] odd) is validated;
        `simplify_cube` checks every transform it is handed, so a configuration
        this does not handle raises there rather than producing a wrong answer.
        """
        arcs = copy.deepcopy(self.arc_data)
        out = self._transformed(arcs)
        arc = {0: self._arc(i)[:4], 1: self._arc(j)[:4]}
        pos = {k: arc[k].index(common_edge) for k in arc}
        nxt = {k: (pos[k] - (-1) ** pos[0]) % 4 for k in arc}
        far = [(2, 3), (2, 3), (0, 1), (0, 1)][pos[1]]
        X = lambda e: sp.Symbol(f"X_{e}")
        X_e0, X_e1 = X(arc[0][3 - pos[0]]), X(arc[1][3 - pos[1]])
        X_e2, X_e3 = X(arc[1][far[0]]), X(arc[1][far[1]])
        arcs[i - 1][pos[0]], arcs[i - 1][nxt[0]] = arc[1][nxt[1]], common_edge
        arcs[j - 1][pos[1]], arcs[j - 1][nxt[1]] = arc[0][nxt[0]], common_edge
        sha = self._arc(i)[4]
        if (pos[0] - pos[1]) % 2 == 0:
            # the tip slide r_{j,i}(x + x' + G) of the arc slide lemma, with one
            # of the two outer edges taken from each arc
            s = ((-1) ** ((pos[0] + pos[1]) // 2 + sha + 1)
                 * (X_e0 + X_e1 + self.G) * T(i) * T(j))
            out.iso, out.iso_inv = (1 + s) * out.iso, out.iso_inv * (1 - s)
        else:
            s = (-1) ** ((pos[0] + pos[1] - 1) // 2) * (
                T(i) * D(j) + (-1) ** sha * (X_e2 - X_e3) * T(i) * T(j))
            out.iso, out.iso_inv = (1 + s) * out.iso, out.iso_inv * (1 - s)
        return out

# ======================================================================
# 4.  Cubes of resolutions
# ======================================================================

def resolve_crossing(crossing, bit, parity=0):
    """The arc of one resolved crossing.  `parity` is the crossing's position parity."""
    e0, e1, e2, e3, c, s = crossing
    if s not in (1, -1):
        raise ValueError("crossing sign must be 1 or -1")
    oriented = (bit == 0) if s == 1 else (bit == 1)
    if parity == 0:
        return [e3, e0, e1, e2, c] if oriented else [e0, e1, e2, e3, 1 - c]
    return [e1, e2, e3, e0, c] if oriented else [e2, e3, e0, e1, 1 - c]

def get_crossing_map(crossing, a):
    """The saddle D_a +- (X_e0 + X_e2 + G) T_a of the crossing carrying arc a."""
    e0, e1, e2, e3, c, s = crossing
    if c not in (0, 1):
        raise ValueError("crossing colour must be 0 or 1")
    weight = sp.Symbol(f"X_{e0}") + sp.Symbol(f"X_{e2}") + sp.Symbol('G')
    return (-1) ** (a - 1) * (D(a) + (-1) ** c * weight * T(a))

def build_tangle_cube(tangle):
    """{bits: factorization} and {crossing index: saddle map}.

    The resolution of a crossing depends on its POSITION in the list, through
    `parity = index % 2`, and the saddle carries (-1)^index.  The list is
    therefore not permutable: reordering it describes a different diagram.
    """
    vertices = {bits: SymbolicMatrixFactorization(
                    [resolve_crossing(tangle[i], b, i % 2) for i, b in enumerate(bits)])
                for bits in itertools.product([0, 1], repeat=len(tangle))}
    maps = {a: get_crossing_map(c, a) for a, c in enumerate(tangle, start=1)}
    return vertices, maps

def _nbrs(v, n):
    """(arc a, neighbour) for each 0 bit of v; the a-th bit carries arc a."""
    for a in range(1, n + 1):
        if v[a - 1] == 0:
            yield a, tuple(v[k] + int(k == a - 1) for k in range(n))

def simplify_cube(tangle, gens, gens_simp, u_edge, simplification_type, transforms=None):
    """
    Build the cube, apply the Step-1 isomorphisms, simplify every vertex by an
    explicit special deformation retract, and return the length-1 differential
    the perturbation lemma induces on the simplified cube.

    tangle               crossings [e0, e1, e2, e3, colour, sign]
    gens                 generators of the edge ring, u_edge among them
    gens_simp            generators of the boundary ring
    u_edge               the internal edge playing the role of u
    simplification_type  {vertex: (arc, 1 = arc removal | 2 = delooping)}
    transforms           {vertex: callable(mf) -> mf}, applied before simplifying
    """
    n = len(tangle)
    cube = list(itertools.product([0, 1], repeat=n))
    X_u = sp.Symbol(f"X_{u_edge}")
    vertices, maps = build_tangle_cube(tangle)
    maps = {i: m.subs(vertices[cube[0]].get_custom_ring_reduction(gens))
            for i, m in maps.items()}

    # Step 1, with each recorded isomorphism verified: iso must be invertible and
    # must intertwine the two differentials.  Transforms that do not compose the
    # way they were meant to (a relabeling before a slide, say) fail here rather
    # than quietly poisoning every arrow into the vertex.
    for v, fn in (transforms or {}).items():
        before, after = vertices[v], fn(vertices[v])
        red = before.get_custom_ring_reduction(gens)
        iso = group_and_factor(sp.expand(after.iso).subs(red))
        inv = group_and_factor(sp.expand(after.iso_inv).subs(red))
        if sp.expand(group_and_factor(iso * inv).subs(red)) != 1:
            raise ValueError(f"transform at {v}: iso is not invertible")
        if sp.expand(group_and_factor(
                iso * before.get_differential(gens)
                - after.get_differential(gens) * iso).subs(red)) != 0:
            raise ValueError(f"transform at {v}: iso is not an isomorphism of "
                             "matrix factorizations")
        vertices[v] = after
    subs_edge = {v: vertices[v].get_custom_ring_reduction(gens) for v in cube}

    simp, incl, subs_simp = {}, {}, {}
    for v in cube:
        arc, deg = simplification_type[v]
        base = vertices[v].get_inclusion_operator(arc, u_edge, deg, subs_edge[v])
        reduced = vertices[v].get_reduction(arc, deg)
        for k in ([()] if deg == 1 else [(0,), (1,)]):
            simp[v + k] = reduced
            incl[v + k] = group_and_factor(
                vertices[v].iso_inv * (X_u ** k[0] if k else 1) * base)
        subs_simp[v] = reduced.get_custom_ring_reduction(gens_simp)

    linear = [v for v in cube if simplification_type[v][1] == 1]
    quadratic = [v for v in cube if simplification_type[v][1] == 2]
    simp_domain = linear + [v + (k,) for v in quadratic for k in (0, 1)]

    d1 = {}
    for v in simp_domain:
        for a, w in _nbrs(v[:n], n):
            arc_v, arc_w = simplification_type[v[:n]][0], simplification_type[w][0]
            kill = {D(arc_v): 0, T(arc_w): 0}
            raw = group_and_factor(vertices[w].iso * maps[a] * incl[v])
            if w in linear:
                targets = [(w, group_and_factor(D(arc_w) * T(arc_w) * raw))]
            else:
                div = vertices[w].get_quadratic(arc_w).subs(subs_edge[w])
                targets = [(w + (k,), P(group_and_factor(D(arc_w) * raw), div, X_u, k))
                           for k in (0, 1)]
            for tgt, expr in targets:
                d1[v, tgt] = expr.subs(subs_simp[w]).subs(kill)

    return dict(vertices=vertices, maps=maps, simp=simp, incl=incl,
                subs_edge=subs_edge, subs_simp=subs_simp, d1=d1,
                simp_domain=simp_domain, linear=linear, quadratic=quadratic,
                cube=cube, u=X_u, n=n, simplification_type=simplification_type)

# ======================================================================
# 5.  Length-2 component of the perturbed differential
# ======================================================================

def arc_of(res, v):
    return res["simplification_type"][v[:res["n"]]][0]

def idem(res, v):
    """The identity of the simplified vertex v, as the idempotent d_a t_a."""
    a = arc_of(res, v)
    return D(a) * T(a)

def _kill(res, v, w):
    return {D(arc_of(res, v)): 0, T(arc_of(res, w)): 0}

def _lift(res, bv):
    return [bv] if bv in res["linear"] else [bv + (0,), bv + (1,)]

def _kappa(res, m, expr):
    """kappa_m applied to expr, without the overall minus sign."""
    a, mf = arc_of(res, m), res["vertices"][m]
    if m in res["linear"]:
        op, div = T(a), mf.get_linear(a)
    else:
        op, div = D(a), mf.get_quadratic(a)
    return Quot(group_and_factor(op * expr), div.subs(res["subs_edge"][m]), res["u"])

def d2(res, v, w):
    """
    -sum_m rho_w . map_j . kappa_m . map_i . iota_v over the two intermediate
    vertices m.  A delooped target is projected with P^{u,l} . d_j, a linearly
    reduced one with the idempotent d_a t_a.
    """
    n = res["n"]
    bv, bw = v[:n], w[:n]
    diff = [k + 1 for k in range(n) if bw[k] != bv[k]]
    assert len(diff) == 2 and all(bv[a - 1] == 0 for a in diff)
    total = 0
    for a, b in [(diff[0], diff[1]), (diff[1], diff[0])]:
        bm = tuple(bv[k] + int(k == a - 1) for k in range(n))
        inner = group_and_factor(res["vertices"][bm].iso * res["maps"][a] * res["incl"][v])
        body = group_and_factor(-res["vertices"][bw].iso * res["maps"][b]
                                * res["vertices"][bm].iso_inv * _kappa(res, bm, inner))
        if bw in res["quadratic"]:
            a_w = arc_of(res, w)
            div = res["vertices"][bw].get_quadratic(a_w).subs(res["subs_edge"][bw])
            total += P(group_and_factor(D(a_w) * body), div, res["u"], w[n])
        else:
            total += group_and_factor(idem(res, w) * body)
    return group_and_factor(total).subs(res["subs_simp"][bw]).subs(_kill(res, v, w))

def all_d2(res):
    n, out = res["n"], {}
    for v in res["simp_domain"]:
        bv = v[:n]
        for i, j in itertools.combinations([k for k in range(n) if bv[k] == 0], 2):
            bw = tuple(bv[k] + int(k in (i, j)) for k in range(n))
            for w in _lift(res, bw):
                e = d2(res, v, w)
                if e != 0:
                    out[v, w] = e
    return out

# ======================================================================
# 6.  Gaussian elimination
#
# A simplified vertex is free of rank 2^(r-1) on the monomials in its
# surviving arcs, so an arrow between two of them has a matrix over the
# boundary ring.  That is what decides invertibility: an arrow is an
# isomorphism exactly when its determinant is a unit.  Testing
# "psi * e == 1" in the free algebra instead would be one sided and
# meaningless when the two ends are simplified at different arcs.
# ======================================================================

@functools.lru_cache(maxsize=None)
def _wedge_ops(r):
    """Matrices of T_1..T_r and D_1..D_r on the exterior algebra on r generators."""
    arcs = range(1, r + 1)
    sub = [frozenset(s) for k in range(r + 1) for s in itertools.combinations(arcs, k)]
    idx = {s: i for i, s in enumerate(sub)}
    def mat(a, raise_):
        M = sp.zeros(len(sub), len(sub))
        for s in sub:
            if raise_ == (a in s):
                continue
            t = s | {a} if raise_ else s - {a}
            M[idx[t], idx[s]] = (-1) ** len([b for b in s if b < a])
        return M
    return (sub, idx, {a: mat(a, True) for a in arcs}, {a: mat(a, False) for a in arcs})

def act(e, r):
    """The matrix by which an expression acts on the exterior algebra on r arcs."""
    sub, _, TH, DD = _wedge_ops(r)
    M = sp.zeros(len(sub), len(sub))
    for term in sp.Add.make_args(sp.expand(group_and_factor(e))):
        c, ops = parse_nc_term(term)
        P_ = sp.eye(len(sub))
        for o in ops:
            P_ = P_ * (TH if o.name[0] == 'T' else DD)[int(o.name[2:])]
        M += c * P_
    return M

def matrix_of(res, A, B, e):
    """The matrix over the boundary ring of an arrow e : M_A -> M_B."""
    r = len(res["simp"][A].arc_data)
    sub, idx, _, _ = _wedge_ops(r)
    keep = lambda v: [idx[s] for s in sub if arc_of(res, v) not in s]
    M, ca, cb = act(e, r), keep(A), keep(B)
    return sp.Matrix(len(cb), len(ca), lambda i, j: sp.expand(M[cb[i], ca[j]]))

def _matrix_units(res, B, A):
    """Expressions whose matrices are the matrix units of Hom(M_B, M_A)."""
    r = len(res["simp"][A].arc_data)
    sub, _, _, _ = _wedge_ops(r)
    pi = sp.Mul(*[D(a) * T(a) for a in range(1, r + 1)])   # onto the constant monomial
    out = []
    for t in [s for s in sub if arc_of(res, A) not in s]:
        for s in [s for s in sub if arc_of(res, B) not in s]:
            out.append(group_and_factor(sp.Mul(*[T(a) for a in sorted(t)]) * pi
                                        * sp.Mul(*[D(a) for a in sorted(s, reverse=True)])))
    return out

def invert(res, A, B, e):
    """Two-sided inverse of the arrow e : M_A -> M_B, or None if it is not one."""
    N = matrix_of(res, A, B, e)
    if sp.expand(N.det()) not in (sp.Integer(1), sp.Integer(-1)):
        return None
    N_inv, psi = sp.expand(N.inv()), sp.S.Zero
    for u in _matrix_units(res, B, A):
        M = matrix_of(res, B, A, u)
        nz = [(i, j) for i in range(M.rows) for j in range(M.cols) if M[i, j] != 0]
        assert len(nz) == 1, "matrix units are not matrix units"
        i, j = nz[0]
        psi += N_inv[i, j] / M[i, j] * u
    return group_and_factor(psi)

def cancel(res, d, verts, A, B, phi_inv):
    """Remove the 1-contractible summand {A, B} along the isomorphism d[A, B]."""
    out = {k: e for k, e in d.items() if not {k[0], k[1]} & {A, B}}
    for (v, wB), e1 in d.items():
        if wB != B or v in (A, B):
            continue
        for (vA, w), e2 in d.items():
            if vA != A or w in (A, B):
                continue
            corr = group_and_factor(e2 * idem(res, A) * phi_inv * e1).subs(_kill(res, v, w))
            if corr != 0:
                out[v, w] = group_and_factor(out.get((v, w), 0) - corr).subs(_kill(res, v, w))
    return ({k: e for k, e in out.items() if e != 0},
            [x for x in verts if x not in (A, B)])

# ======================================================================
# 7.  Verification and display
# ======================================================================

def check_chain_maps(res, d, gens, label):
    bad = [(v, w, chk) for (v, w), e in d.items()
           if (chk := sp.expand(hom_differential(
               e, res["simp"][v].get_differential(gens),
               res["simp"][w].get_differential(gens), 1).subs(_kill(res, v, w)))) != 0]
    print(f"  {label}: mu_1 = 0 for {len(d) - len(bad)}/{len(d)} arrows")
    for v, w, chk in bad:
        print("     ", v, "->", w, ":", chk)
    return bad

def check_potential(res, verts, gens, label):
    sq = {sp.expand(group_and_factor(res["simp"][v].get_differential(gens) ** 2))
          for v in verts}
    print(f"  {label}: d_0^2 equal at all {len(verts)} vertices: {len(sq) == 1}")
    return sq.pop() if len(sq) == 1 else None

def check_h2(res, d, verts, gens, label):
    """d_1 d_1 + mu_1(d_2) = 0 for every pair of vertices two steps apart."""
    h = lambda t: sum(t[:res["n"]])
    pairs = [(v, w) for v in verts for w in verts if h(w) - h(v) == 2]
    bad = 0
    for v, w in pairs:
        comp = sum(group_and_factor(d[m, w] * idem(res, m) * d[v, m])
                   for m in verts if (v, m) in d and (m, w) in d and h(m) - h(v) == 1)
        mu = hom_differential(d[v, w], res["simp"][v].get_differential(gens),
                              res["simp"][w].get_differential(gens)) if (v, w) in d else 0
        if sp.expand(group_and_factor(comp + mu).subs(_kill(res, v, w))) != 0:
            bad += 1
    print(f"  {label}: d_1 d_1 + mu_1(d_2) = 0 for {len(pairs) - bad}/{len(pairs)} pairs")
    return bad

def show_vertices(res, verts, gens, title):
    print("=" * 72); print(title); print("=" * 72)
    for v in sorted(verts):
        print(f"  {v}   arcs {res['simp'][v].arc_data}")
        print(f"        d_0 = {res['simp'][v].get_differential(gens)}")
    print()

def show_arrows(d, title, n=3, length=None):
    h = lambda t: sum(t[:n])
    print("=" * 72); print(title); print("=" * 72)
    keys = [k for k in d if length is None or h(k[1]) - h(k[0]) == length]
    for k in sorted(keys, key=lambda t: (h(t[0]), t[0], t[1])):
        print(f"  {k[0]} --> {k[1]}   ({h(k[1]) - h(k[0])}):  {d[k]}")
    if not keys:
        print("  (none)")
    print()

# ======================================================================
# 8.  Symbols used by the sections below
# ======================================================================

X_0, X_1, X_2, X_3, X_4, X_5, X_6, X_7, X_8 = sp.symbols(
    'X_0 X_1 X_2 X_3 X_4 X_5 X_6 X_7 X_8')
T_1, T_2, T_3 = sp.symbols('T_1 T_2 T_3', commutative=False)
D_1, D_2, D_3 = sp.symbols('D_1 D_2 D_3', commutative=False)
G = sp.Symbol('G')

CUBE2 = list(itertools.product([0, 1], repeat=2))
CUBE3 = list(itertools.product([0, 1], repeat=3))
G3 = [X_0, X_1, X_2]                     # boundary ring for R1, R2
G5 = [X_0, X_1, X_2, X_3, X_4]           # boundary ring for R3
h = lambda t: sum(t[:3])                 # homological degree of a cube vertex

## Reidemeister I

One crossing, both signs. Delooping at the $0$-resolution and arc removal at the
$1$-resolution leave a single summand.

In [2]:
for sign, st in [(+1, {(0,): (1, 2), (1,): (1, 1)}),
                 (-1, {(0,): (1, 1), (1,): (1, 2)})]:
    r = simplify_cube(tangle=[[0, 1, 2, 2, 0, sign]],
                      gens=[X_0, X_1], gens_simp=[X_0], u_edge=2,
                      simplification_type=st)
    lbl = f'R1 (crossing sign {sign:+d})'
    show_vertices(r, r['simp_domain'], [X_0], lbl + ': vertices')
    show_arrows(r['d1'], lbl + ': arrows', n=1)
    check_chain_maps(r, r['d1'], [X_0], lbl)
    iso = [k for k, e in r['d1'].items() if invert(r, k[0], k[1], e) is not None]
    print(f'  invertible component, the one Lemma SummandLemma removes: {iso}')
    print()

R1 (crossing sign +1): vertices
  (0, 0)   arcs [[2, 0, 1, 2, 3]]
        d_0 = 0
  (0, 1)   arcs [[2, 0, 1, 2, 3]]
        d_0 = 0
  (1,)   arcs [[0, 1, 2, 2, 2]]
        d_0 = 0

R1 (crossing sign +1): arrows
  (0, 0) --> (1,)   (1):  1
  (0, 1) --> (1,)   (1):  X_0

  R1 (crossing sign +1): mu_1 = 0 for 2/2 arrows
  invertible component, the one Lemma SummandLemma removes: [((0, 0), (1,))]

R1 (crossing sign -1): vertices
  (0,)   arcs [[0, 1, 2, 2, 2]]
        d_0 = 0
  (1, 0)   arcs [[2, 0, 1, 2, 3]]
        d_0 = 0
  (1, 1)   arcs [[2, 0, 1, 2, 3]]
        d_0 = 0

R1 (crossing sign -1): arrows
  (0,) --> (1, 0)   (1):  G + X_0
  (0,) --> (1, 1)   (1):  1

  R1 (crossing sign -1): mu_1 = 0 for 2/2 arrows
  invertible component, the one Lemma SummandLemma removes: [((0,), (1, 1))]



## Reidemeister II

Two crossings of opposite sign; edge `4` is $u$. The arc simplified at each
vertex is forced -- the linear term has to be monic in $u$ -- and the table
printed first records that, which is why the vertex `(1,0)` is delooped rather
than reduced. R2b is R2a with every arc reversed, as in the text, so it is the
same cube with the arc reversal $s_a$ applied at each vertex; the last line
checks that this turns every arrow into its negative. The length-2 arrows vanish
in both, which is the claim in the proof that Formula (deformed sdr data) could
in principle produce them here but does not.

In [3]:
TAN_R2 = [[0, 1, 5, 4, 0, 1], [2, 3, 4, 5, 0, -1]]
ST_R2 = {(0, 0): (2, 1), (1, 0): (2, 1), (1, 1): (1, 1), (0, 1): (2, 2)}

_v, _ = build_tangle_cube(TAN_R2)
print('which arc carries u at each vertex of R2a')
for _x in sorted(_v):
    _s = _v[_x].get_custom_ring_reduction([X_0, X_1, X_2, X_4])
    _d = {a: sp.degree(sp.expand(_v[_x].get_linear(a).subs(_s)), X_4)
          for a in _v[_x].arcs}
    _ok = [a for a, e in _d.items() if e == 1]
    print(f'   {_x}  deg_u(linear) = {_d}  ->',
          f'arc removal at {_ok}' if _ok else 'none: must deloop')
print()

R2 = {}
for lbl, tr in [('R2a', None),
                ('R2b', {v: (lambda m: m.flip_arrows()) for v in CUBE2})]:
    r = simplify_cube(tangle=TAN_R2, gens=[X_0, X_1, X_2, X_4], gens_simp=G3,
                      u_edge=4, simplification_type=ST_R2, transforms=tr)
    R2[lbl] = r
    show_vertices(r, r['simp_domain'], G3, lbl + ': vertices')
    show_arrows(r['d1'], lbl + ': arrows', n=2)
    check_chain_maps(r, r['d1'], G3, lbl)
    D2 = all_d2(r)
    print(f'  length-2 arrows: {D2 if D2 else "all vanish, as claimed"}')
    assert not D2
    iso = [k for k, e in r['d1'].items() if invert(r, k[0], k[1], e) is not None]
    print(f'  invertible components, the ones Lemma SummandLemma removes: {iso}')
    print()

print('  every arrow of R2b is minus the corresponding arrow of R2a:',
      all(sp.expand(group_and_factor(R2['R2b']['d1'][k] + e)) == 0
          for k, e in R2['R2a']['d1'].items()))

which arc carries u at each vertex of R2a
   (0, 0)  deg_u(linear) = {1: 0, 2: 1}  -> arc removal at [2]
   (0, 1)  deg_u(linear) = {1: 0, 2: 0}  -> none: must deloop
   (1, 0)  deg_u(linear) = {1: 1, 2: 1}  -> arc removal at [1, 2]
   (1, 1)  deg_u(linear) = {1: 1, 2: 0}  -> arc removal at [1]

R2a: vertices
  (0, 0)   arcs [[4, 0, 1, 5, 0], [4, 5, 2, 3, 2]]
        d_0 = (X_0 - X_1)*D_1 + (X_1 - X_2)*(G + X_0 + X_2)*T_1
  (0, 1, 0)   arcs [[1, 5, 4, 0, 0], [3, 4, 5, 2, 3]]
        d_0 = (-X_0 + X_1)*D_1 - (X_1 - X_2)*(G + X_0 + X_2)*T_1
  (0, 1, 1)   arcs [[1, 5, 4, 0, 0], [3, 4, 5, 2, 3]]
        d_0 = (-X_0 + X_1)*D_1 - (X_1 - X_2)*(G + X_0 + X_2)*T_1
  (1, 0)   arcs [[0, 1, 5, 4, 1], [4, 5, 2, 3, 2]]
        d_0 = -(X_0 - X_1)*(G + X_0 + X_2)*T_1 + (-X_1 + X_2)*D_1
  (1, 1)   arcs [[0, 1, 5, 4, 2], [3, 4, 5, 2, 0]]
        d_0 = (X_0 - X_1)*D_2 - (-X_1 + X_2)*(G + X_0 + X_2)*T_2

R2a: arrows
  (0, 0) --> (0, 1, 0)   (1):  -G - X_2
  (0, 0) --> (0, 1, 1)   (1):  -1
  (0, 0) --> (1,

## Reidemeister III, Steps 1 and 2

Three crossings; edge `7` is $u$. `transforms` is Step 1: an arc slide at
`(1,0,1)` and label permutations at `(1,1,0)` and `(1,1,1)`, each an isomorphism,
recorded in `.iso` and carried through -- so the cube built here is $M'_{R3}$.
`simplification_type` is Step 2: an arc removal at seven vertices and a delooping
at `(1,0,1)`, which splits that vertex in two. The perturbation lemma then gives
$M''_{R3}$, whose length-1 and length-2 arrows are printed.

`M` is the left-hand side of the Reidemeister 3 equivalence and `N` the
right-hand side, set up with the same arcs simplified at the corresponding
vertices.

In [4]:
R3 = {}

trM = {v: (lambda m: m) for v in CUBE3}
trM[(1, 0, 1)] = lambda m: m.arc_slide(2, 3, 8)
trM[(1, 1, 0)] = lambda m: m.arc_relabel({1: 3})
trM[(1, 1, 1)] = lambda m: m.arc_relabel({2: 3})
R3['M'] = dict(
    tangle=[[5, 6, 7, 4, 1, 1], [0, 1, 8, 6, 0, 1], [8, 2, 3, 7, 1, 1]],
    transforms=trM,
    simplification_type={(0, 0, 0): (1, 1), (0, 0, 1): (1, 1),
                         (0, 1, 0): (1, 1), (0, 1, 1): (1, 1),
                         (1, 0, 0): (3, 1), (1, 0, 1): (3, 2),
                         (1, 1, 0): (1, 1), (1, 1, 1): (3, 1)},
    cancellations=[((1, 0, 0), (1, 0, 1, 1)), ((1, 0, 1, 0), (1, 1, 1))])

trN = {v: (lambda m: m) for v in CUBE3}
trN[(1, 1, 0)] = lambda m: m.arc_relabel({1: 3})
trN[(1, 1, 1)] = lambda m: m.arc_relabel({2: 3})
trN[(1, 0, 1)] = lambda m: m.arc_slide(2, 3, 6)
R3['N'] = dict(
    tangle=[[7, 1, 2, 8, 0, 1],[6, 8, 3, 4, 1, 1], [0, 7, 6, 5, 0, 1]],
    transforms=trN,
    simplification_type={(0, 0, 0): (1, 1), (0, 0, 1): (1, 1),
                         (0, 1, 0): (1, 1), (0, 1, 1): (1, 1),
                         (1, 0, 0): (3, 1), (1, 1, 0): (1, 1),
                         (1, 0, 1): (3, 2), (1, 1, 1): (3, 1)},
    cancellations=[((1, 0, 0), (1, 0, 1, 1)), ((1, 0, 1, 0), (1, 1, 1))])

for side, cfg in R3.items():
    r = simplify_cube(tangle=cfg['tangle'], gens=[X_0, X_1, X_2, X_3, X_4, X_7],
                      gens_simp=G5, u_edge=7,
                      simplification_type=cfg['simplification_type'],
                      transforms=cfg['transforms'])
    r['d2'] = all_d2(r)
    cfg['res'] = r
    print(f"--- {side}''_R3 ---")
    show_arrows(r['d1'], f"{side}''_R3: length-1 arrows")
    show_arrows(r['d2'], f"{side}''_R3: length-2 arrows")
    check_chain_maps(r, r['d1'], G5, f"{side}''_R3")
    print()

--- M''_R3 ---
M''_R3: length-1 arrows
  (0, 0, 0) --> (0, 0, 1)   (1):  (-G - X_2 - X_4)*T_3 + D_3
  (0, 0, 0) --> (0, 1, 0)   (1):  (-G - X_0 - X_2 + X_3 - X_4)*T_2 - D_2
  (0, 0, 0) --> (1, 0, 0)   (1):  (-G - X_0 + X_1 - X_2 - X_4)*T_1 + (-G - X_0 + X_1 - 2*X_2 + X_3 - X_4)*T_2 + D_3
  (0, 0, 1) --> (0, 1, 1)   (1):  (-G - X_0 - X_2 + X_3 - X_4)*T_2 - D_2
  (0, 0, 1) --> (1, 0, 1, 0)   (1):  G + X_2 + X_3*(X_3 - X_4)*T_1*T_2 - X_3 + X_4 + (G + X_0 - X_1 + X_2 - X_3 + X_4)*T_1*D_3 + (G + X_0 - X_1 + 2*X_2 - 2*X_3 + X_4)*T_2*D_3
  (0, 0, 1) --> (1, 0, 1, 1)   (1):  (-X_3 + X_4)*T_1*T_2 + 1 + T_1*D_3 + T_2*D_3
  (0, 1, 0) --> (0, 1, 1)   (1):  (-G - X_2 - X_4)*T_3 + D_3
  (0, 1, 0) --> (1, 1, 0)   (1):  (-X_0 + X_1)*T_2 + (X_0 - X_1)*T_2*T_3*D_3 + (-G - X_0 + X_1 - X_2 - X_4)*T_3 + D_2 - D_2*T_3*D_3 + D_3
  (1, 0, 0) --> (1, 0, 1, 0)   (1):  -G - X_2
  (1, 0, 0) --> (1, 0, 1, 1)   (1):  -1
  (1, 0, 0) --> (1, 1, 0)   (1):  (-G - X_0 - X_2)*D_1*T_2*T_3 + (-G - X_0 - X_2)*T_2 - D_1*D_2*

## Reidemeister III, Step 3

Two summands are removed by Gaussian elimination. The length-1 and length-2
arrows are carried through together, so the reduced complex keeps its higher
components. `invert` decides invertibility from the $4\times4$ matrix of the
cancelling arrow over the boundary ring, so it is two-sided and stays correct
when the two ends are simplified at different arcs. The results are the
multifactorizations $M'''_{R3}$ and $N'''_{R3}$ of the final figure.

In [5]:
for side, cfg in R3.items():
    r = cfg['res']
    d, verts = {**r['d1'], **r['d2']}, list(r['simp_domain'])
    for A, B in cfg['cancellations']:
        phi = r['d1'][A, B]
        inv = invert(r, A, B, phi)
        assert inv is not None, f'{A}->{B} is not invertible'
        print(f"  {side}''_R3: cancelling {A} -> {B},  phi = {phi}")
        d, verts = cancel(r, d, verts, A, B, inv)
    cfg['final'] = (d, verts)
    print()
    show_vertices(r, verts, G5, f"{side}'''_R3: vertices and d_0")
    show_arrows(d, f"{side}'''_R3: length-1 arrows", length=1)
    show_arrows(d, f"{side}'''_R3: length-2 arrows", length=2)
    check_potential(r, verts, G5, f"{side}'''_R3")
    check_chain_maps(r, {k: e for k, e in d.items() if h(k[1]) - h(k[0]) == 1},
                     G5, f"{side}'''_R3")
    check_h2(r, d, verts, G5, f"{side}'''_R3")
    print()

  M''_R3: cancelling (1, 0, 0) -> (1, 0, 1, 1),  phi = -1
  M''_R3: cancelling (1, 0, 1, 0) -> (1, 1, 1),  phi = (-X_1 + X_2)*T_1*T_2 + 1 - T_1*D_2

M'''_R3: vertices and d_0
  (0, 0, 0)   arcs [[4, 5, 6, 7, 2], [1, 8, 6, 0, 0], [7, 8, 2, 3, 1]]
        d_0 = (-X_0 + X_1)*D_2 - (-X_2 + X_3)*(G + X_2 + X_4)*T_3 + (X_3 - X_4)*D_3 - (X_1 - X_2 + X_3 - X_4)*(G + X_0 + X_2 - X_3 + X_4)*T_2
  (0, 0, 1)   arcs [[4, 5, 6, 7, 2], [1, 8, 6, 0, 0], [8, 2, 3, 7, 0]]
        d_0 = (-X_0 + X_1)*D_2 + (X_2 - X_3)*D_3 + (X_3 - X_4)*(G + X_2 + X_4)*T_3 - (X_1 - X_2 + X_3 - X_4)*(G + X_0 + X_2 - X_3 + X_4)*T_2
  (0, 1, 0)   arcs [[4, 5, 6, 7, 2], [8, 6, 0, 1, 1], [7, 8, 2, 3, 1]]
        d_0 = (X_0 - X_1)*(G + X_0 + X_2 - X_3 + X_4)*T_2 - (-X_2 + X_3)*(G + X_2 + X_4)*T_3 + (X_3 - X_4)*D_3 + (X_1 - X_2 + X_3 - X_4)*D_2
  (0, 1, 1)   arcs [[4, 5, 6, 7, 2], [8, 6, 0, 1, 1], [8, 2, 3, 7, 0]]
        d_0 = (X_0 - X_1)*(G + X_0 + X_2 - X_3 + X_4)*T_2 + (X_2 - X_3)*D_3 + (X_3 - X_4)*(G + X_2 + X_4)*T_3 + (X_1 